# set up environment

### initialize

In [1]:
import os
import shutil
import glob
import pickle
import warnings
warnings.filterwarnings("ignore")
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.ion()
from datetime import datetime
from Toolshed import Download, Toolbox, VegetationLine, Plotting, PlottingSeaborn, Transects
import ee
import geopandas as gpd
import geemap
from shapely.geometry import MultiPolygon
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

ee.Initialize()
ee.Authenticate() # should only need to be run the first time after installation

True

### parameter settings

In [2]:
# The points represent the corners of a bounding box that go around your site
sitename = 'ConnorsCove_BlackBeach'
# ConnorsCove_BlackBeach

# Date range
dates = ['2020-01-01', '2025-06-01']

# Satellite missions
# Input a list of containing any/all of 'L5', 'L7', 'L8', 'L9', 'S2', 'PSScene4Band'
# L5: 1984-2013; L7: 1999-2017 (SLC error from 2003); L8: 2013-present; S2: 2014-present; L9: 2021-present
sat_list = ['PSScene4Band']

# Cloud threshold for screening out cloudy imagery (0.5 or 50% recommended)
cloud_thresh = 0.3

# Extract shoreline (wet-dry boundary) as well as veg edge
wetdry = True

# Directory where the data will be stored
filepath = Toolbox.CreateFileStructure(sitename, sat_list)

# load locations information and Reference Lines

### load reference shoreline AOI on reference shoreline (in library)

In [3]:
# Reference shoreline/veg line shapefile name (should be stored in a folder called referenceLines in Data)
# Line should be ONE CONTINUOUS linestring along the shore, stored as a shapefile in WGS84 coord system
referenceLineShp = sitename + '_ref.shp'
# Maximum amount in metres by which to buffer the reference line for capturing veg edges within
max_dist_ref = 150

In [4]:
# get AOI from file
polygon, point, gdf, lonmin, lonmax, latmin, latmax = Toolbox.AOI_from_file(sitename)

# if using referenceline to generate ROI
# polygon, point, lonmin, lonmax, latmin, latmax = Toolbox.AOIfromLine(referenceLinePath, max_dist_ref, sitename)

In [5]:
# Return AOI from reference line bounding box and save AOI folium map HTML in sitename directory
referenceLinePath = os.path.join(filepath, 'referenceLines', referenceLineShp)
referenceLineDF = gpd.read_file(referenceLinePath)

In [6]:
referenceLinePath

'/home/sagemaker-user/COASTGUARD/Data/referenceLines/ConnorsCove_BlackBeach_ref.shp'

### date range

In [7]:
if len(dates)>2:
    daterange='no'
else:
    daterange='yes'
years = list(Toolbox.daterange(datetime.strptime(dates[0],'%Y-%m-%d'), datetime.strptime(dates[-1],'%Y-%m-%d')))

### Compile Input Settings for Gathering Imagery

In [8]:
inputs = {
    'polygon': polygon,
    'dates': dates,
    'daterange': daterange,
    'sat_list': sat_list,
    'sitename': sitename,
    'filepath': filepath
}

In [9]:
inputs

{'polygon': [[[-66.24266461754597, 45.137203474471825],
   [-66.2450805663957, 45.1677522829498],
   [-66.24529327738861, 45.16793094202593],
   [-66.24790489868946, 45.18180590894927],
   [-66.2590274537731, 45.18695469974523],
   [-66.25920456295579, 45.18885695625626],
   [-66.26084604561237, 45.189872635386344],
   [-66.25226146021818, 45.19893538546543],
   [-66.24612657, 45.19764772],
   [-66.22578366, 45.19501472],
   [-66.20876204, 45.18901687],
   [-66.20417504, 45.161377],
   [-66.21490662, 45.14377346],
   [-66.21625861, 45.1392437],
   [-66.24266461754597, 45.137203474471825]]],
 'dates': ['2020-01-01', '2025-06-01'],
 'daterange': 'yes',
 'sat_list': ['PSScene4Band'],
 'sitename': 'ConnorsCove_BlackBeach',
 'filepath': '/home/sagemaker-user/COASTGUARD/Data'}

### Image Retrieval and download

In [10]:
# inputs = Download.check_images_available(inputs)
Sat = Download.LocalImageRetrieval(inputs)

In [11]:
# Remove the first two empty lists manually
Sat = [s for s in Sat if len(s) > 0]

In [12]:
Sat

[['./Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201012_143405_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201110_152523_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201119_143635_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201122_143553_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210110_143026_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210125_143611_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210131_143736_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210209_152233_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210212_152411_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210226_151924_composite.tif',
  './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210306_152017_composite.tif',

In [13]:
metadata = {}

for i in range(len(inputs['sat_list'])):
    metadata[inputs['sat_list'][i]] = {
        'filenames': [], 
        'acc_georef': [], 
        'epsg': [], 
        'dates': [],
        'tide_level': []
    }

In [14]:
metadata = Download.LocalImageMetadata(inputs, Sat)

PSScene4Band: 100.00%

In [15]:
print(metadata.keys())  # Should print: dict_keys(['PlanetScope'])
print(metadata['PSScene4Band'].keys())  # Should print: dict_keys(['filenames', 'acc_georef', 'epsg', 'dates', 'tide_level'])

dict_keys(['PSScene4Band'])
dict_keys(['filenames', 'acc_georef', 'epsg', 'dates', 'tide_level'])


In [16]:
print(metadata)

{'PSScene4Band': {'filenames': ['./Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201012_143405_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201110_152523_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201119_143635_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201122_143553_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210110_143026_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210125_143611_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210131_143736_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210209_152233_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210212_152411_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210226_151924_composite.tif', './Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20210306_152017_compo

# Veg detection

### settings

In [17]:
BasePath = 'Data/' + sitename + '/lines'

if not os.path.isdir(BasePath):
    os.makedirs(BasePath)

projection_epsg, _ = Toolbox.FindUTM(polygon[0][0][1],polygon[0][0][0])

In [18]:
settings = {
    # general parameters:
    'cloud_thresh': cloud_thresh,        # threshold on maximum cloud cover
    'output_epsg': projection_epsg,     # epsg code of spatial reference system desired for the output   
    'wetdry': wetdry,              # extract wet-dry boundary as well as veg
    # quality control:
    'check_detection': False,    # if True, shows each shoreline detection to the user for validation
    'adjust_detection': False,  # if True, allows user to adjust the postion of each shoreline by changing the threhold
    'save_figure': True,        # if True, saves a figure showing the mapped shoreline for each image
    # [ONLY FOR ADVANCED USERS] shoreline detection parameters:
    'min_beach_area': 10,     # minimum area (in metres^2) for an object to be labelled as a beach
    'buffer_size': 150,         # radius (in metres) for buffer around sandy pixels considered in the shoreline detection
    'min_length_sl': 500,       # minimum length (in metres) of shoreline perimeter to be valid
    'cloud_mask_issue': False,  # switch this parameter to True if sand pixels are masked (in black) on many images  
    # add the inputs defined previously
    'inputs': inputs,
    'projection_epsg': projection_epsg,
    'year_list': years,
}

# Vegetation Line Extraction

##### first time detection: 

In [19]:
referenceLine, ref_epsg = Toolbox.ProcessRefline(referenceLinePath, settings)
settings['reference_shoreline'] = referenceLine
settings['ref_epsg'] = ref_epsg
# Distance to buffer reference line by (this is in metres)
settings['max_dist_ref'] = max_dist_ref

settings['reference_coreg_im'] = None # leave as None if no coregistration is to be performed

In [20]:
print(settings)

{'cloud_thresh': 0.3, 'output_epsg': 32619, 'wetdry': True, 'check_detection': False, 'adjust_detection': False, 'save_figure': True, 'min_beach_area': 10, 'buffer_size': 150, 'min_length_sl': 500, 'cloud_mask_issue': False, 'inputs': {'polygon': [[[-66.24266461754597, 45.137203474471825], [-66.2450805663957, 45.1677522829498], [-66.24529327738861, 45.16793094202593], [-66.24790489868946, 45.18180590894927], [-66.2590274537731, 45.18695469974523], [-66.25920456295579, 45.18885695625626], [-66.26084604561237, 45.189872635386344], [-66.25226146021818, 45.19893538546543], [-66.24612657, 45.19764772], [-66.22578366, 45.19501472], [-66.20876204, 45.18901687], [-66.20417504, 45.161377], [-66.21490662, 45.14377346], [-66.21625861, 45.1392437], [-66.24266461754597, 45.137203474471825]]], 'dates': ['2020-01-01', '2025-06-01'], 'daterange': 'yes', 'sat_list': ['PSScene4Band'], 'sitename': 'ConnorsCove_BlackBeach', 'filepath': '/home/sagemaker-user/COASTGUARD/Data'}, 'projection_epsg': 32619, 'ye

In [ ]:
output, output_latlon, output_proj = VegetationLine.extract_veglines(metadata, settings, polygon, dates)

Mapping veglines:
Loading tide elevations for all images...
PSScene4Band:   0.538 %  
saving ./Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201012_143405_composite.tif
Unique values in im_classif: [ 0.  1.  2.  3. nan]
 
saving classified ./Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201012_143405_composite.tif
 
saving transition zone of ./Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201012_143405_composite.tif

Saving categorical (veg/non-veg base + water + sand + whitewater) for ./Data/ConnorsCove_BlackBeach/local_images/PSScene4Band/20201012_143405_composite.tif
 -> Saved: /home/sagemaker-user/COASTGUARD/Data/ConnorsCove_BlackBeach/img_files/20201012_143405_composite_CLASSIFIED.tif
Tide at 2020-10-12 14:34:05+00:00 = 7.10 m → high
Total water pixels: 445067
Total sand pixels: 97497
Otsu threshold: -0.376


##### second time load in:

In [ ]:
output, output_latlon, output_proj = Toolbox.ReadOutput(inputs)

# Remove Duplicate Lines
# For images taken on the same date by the same satellite, keep only the longest line
output = Toolbox.RemoveDuplicates(output) 

##### save outcome to local shapefile

In [ ]:
# Save output veglines 
Toolbox.SaveConvShapefiles(output, BasePath, sitename, settings['output_epsg'])
# Save output shorelines if they were generated
if settings['wetdry'] == True:
    Toolbox.SaveConvShapefiles_Water(output, BasePath, sitename, settings['output_epsg'])

# Transects detection

#### basics

##### settings

In [ ]:
# settings
SmoothingWindowSize = 21 
NoSmooths = 100
TransectSpacing = 10
DistanceInland = 100
DistanceOffshore = 100

# Provide average beach slope (tanBeta) for site, for calculating corrected beach widths
# Set to 'None' if you want to use CoastSat.slope to calculate per-transect slopes for correcting with
beachslope = None

##### cross-shorelines transects

In [ ]:
VegBasePath = 'Data/' + sitename + '/lines'
VeglineShp = glob.glob(BasePath+'/*veglines.shp')
VeglineGDF = gpd.read_file(VeglineShp[0])
VeglineGDF = VeglineGDF.sort_values(by='dates') # sort GDF by dates to ensure transect intersects occur in chronological order
VeglineGDF = VeglineGDF.reset_index(drop=True) # reset GDF index after date sorting
if settings['wetdry'] == True:
    WaterlineShp = glob.glob(BasePath+'/*waterlines.shp')
    WaterlineGDF = gpd.read_file(WaterlineShp[0])
    WaterlineGDF = WaterlineGDF.sort_values(by='dates') # as above with VeglineGDF date sorting
    WaterlineGDF = WaterlineGDF.reset_index(drop=True)
# Produces Transects for the reference line
TransectSpec =  os.path.join(BasePath, sitename+'_Transects.shp')

# If transects already exist, load them in
if os.path.isfile(TransectSpec[:-3]+'pkl') is False:
    TransectGDF = Transects.ProduceTransects(settings, SmoothingWindowSize, NoSmooths, TransectSpacing, DistanceInland, DistanceOffshore, VegBasePath, referenceLineShp)
else:
    print('Transects already exist and were loaded')
    with open(TransectSpec[:-3]+'pkl', 'rb') as Tfile: 
        TransectGDF = pickle.load(Tfile)
    
# make new transect intersections folder
if os.path.isdir(os.path.join(filepath, sitename, 'intersections')) is False:
    os.mkdir(os.path.join(filepath, sitename, 'intersections'))

##### veg edges intersection along each transect

In [ ]:
if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_intersects.pkl')):
    print('Transect Intersect GDF exists and was loaded')
    with open(os.path.join
              (filepath , sitename, 'intersections', sitename + '_transect_intersects.pkl'), 'rb') as f:
        TransectInterGDF = pickle.load(f)

else:
    # Get intersections
    TransectInterGDF = Transects.GetIntersections(BasePath, TransectGDF, VeglineGDF)
    # Save newly intersected transects as shapefile
    TransectInterGDF = Transects.SaveIntersections(TransectInterGDF, VeglineGDF, BasePath, sitename)
    # Repopulate dict with intersection distances along transects normalised to transect midpoints
    TransectInterGDF = Transects.CalculateChanges(TransectInterGDF)
    
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDF, f)

##### waterlines intersection along each transect

In [ ]:
if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_water_intersects.pkl')):
    print('Transect Intersect + Water GDF exists and was loaded')
    with open(os.path.join
              (filepath , sitename, 'intersections', sitename + '_transect_water_intersects.pkl'), 'rb') as f:
        TransectInterGDFWater = pickle.load(f)
else:        
    if settings['wetdry'] == True:
        TransectInterGDFWater = Transects.GetWaterIntersections(BasePath, TransectGDF, TransectInterGDF, WaterlineGDF, settings, output)  
    
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_water_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDFWater, f)

#### waves and tide included

##### waves intersections

In [ ]:
import traceback

if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_wave_intersects.pkl')):
    print('Transect Intersect + Wave GDF exists and was loaded')
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_wave_intersects.pkl'), 'rb') as f:
        TransectInterGDFWave = pickle.load(f)
else:
    try:
        TransectInterGDFWave = Transects.WavesIntersect(settings, TransectInterGDF, BasePath, output, lonmin, lonmax, latmin, latmax)

        with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_wave_intersects.pkl'), 'wb') as f:
            pickle.dump(TransectInterGDFWave, f)

        print(f'Wave intersect successfully generated and saved for site: {sitename}')

    except Exception as e:
        print(f'Failed to process wave data for site: {sitename}')
        print('Error message:', str(e))
        traceback.print_exc()
        TransectInterGDFWave = None

In [ ]:
# Always recalculate waterline corrections using real slope raster
TransectInterGDFWater = Transects.WLCorrections(
    settings,
    output,
    TransectInterGDFWater,
    TransectInterGDFWave
)

# Calculate the beach width between Vegetation Line and corrected Water Line
TransectInterGDFWater = Transects.CalcBeachWidth(
    settings, 
    TransectGDF, 
    TransectInterGDFWater
)


# Save corrected Water Line intersections shapefile
TransectInterGDFWater = Transects.SaveWaterIntersections(
    TransectInterGDFWater, 
    WaterlineGDF,  
    BasePath, 
    sitename
)

In [ ]:
TransectInterGDFWater.columns

In [ ]:
# Save intersections as a pickle file
with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_water_intersects.pkl'), 'wb') as f:
    pickle.dump(TransectInterGDFWater, f)

plotting

In [ ]:
# Timeseries Plotting

# EDIT ME: Select transect ID to plot
# You can plot subplots within a list of plot IDs, e.g. [[sub1, sub2], plot2]
# You can also comment Line 1 out and uncomment Line 2 to create plots for ALL Transect IDs
# NOTE: If you want to plot ALL transects, it's recommended you switch ShowPlot=False

TransectIDs = [[25,30,35],50,75] # Line 1
# TransectIDs = list(TransectInterGDF['TransectID']) # Line 2

for TransectID in TransectIDs:
    # Plot timeseries of cross-shore veg position
    Plotting.VegTimeseries(sitename, TransectInterGDF, TransectID, Hemisphere='N', ShowPlot=True)
    # If plotting veg and water lines together
    if settings['wetdry']:
        Plotting.VegWaterTimeseries(sitename, TransectInterGDFWater, TransectID, Hemisphere='N', ShowPlot=True)


In [ ]:
TransectID = 0
Plotting.WaterTimeseries(sitename, TransectInterGDFWater, TransectID, Hemisphere='N', ShowPlot=True)

In [ ]:
all_ids = list(range(len(TransectInterGDFWave)))

Plotting.WaveEDAplots(
    sitename,
    TransectInterGDFWave,
    all_ids,
    Hemisphere='N',
    ShowPlot=True
)

In [ ]:
def check_wave_data_lengths(df):
    summary = []
    for i in range(len(df)):
        row = df.iloc[i]
        lengths = {
            'WaveDates': len(row['WaveDates']) if isinstance(row['WaveDates'], list) else 0,
            'WaveHsFD': len(row['WaveHsFD']) if isinstance(row['WaveHsFD'], list) else 0,
            'WaveTpFD': len(row['WaveTpFD']) if isinstance(row['WaveTpFD'], list) else 0,
            'WaveDirFD': len(row['WaveDirFD']) if isinstance(row['WaveDirFD'], list) else 0,
            'Runups': len(row['Runups']) if isinstance(row['Runups'], list) else 0
        }
        summary.append((i, lengths))
    return summary

wave_lengths_summary = check_wave_data_lengths(TransectInterGDFWave)
wave_lengths_summary

In [ ]:
all_ids = list(range(len(TransectInterGDFWater)))
print(len(all_ids))
clustered_df = Plotting.cluster_transects_eda(
    sitename,
    TransectInterGDFWater,
    TransectIDs=all_ids,
    optimal_k=7
)

In [ ]:
# clustered_df = result of cluster_transects_eda
# TransectGDF = full transect shapefile (GeoDataFrame)
m = Plotting.plot_transects_clusters_folium(TransectInterGDFWater, clustered_df, sitename, outfile='clustered_transects_map.html')
m

In [ ]:
# EDIT ME: Path to slope raster for extracting slope values
TIF = './DEM.tif'

if os.path.isfile(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_topo_intersects.pkl')):
    print('Transect Intersect + Topo GDF exists and was loaded')
    with open(os.path.join
              (filepath , sitename, 'intersections', sitename + '_transect_topo_intersects.pkl'), 'rb') as f:
        TransectInterGDFTopo = pickle.load(f)
else:
    # Update Transects with Transition Zone widths and slope if available
    TransectInterGDFTopo = Transects.TZIntersect(settings, TransectInterGDF, VeglineGDF, BasePath)
    TransectInterGDFTopo = Transects.SlopeIntersect(settings, TransectInterGDFTopo, VeglineGDF, BasePath, TIF)
    
    with open(os.path.join(filepath, sitename, 'intersections', sitename + '_transect_topo_intersects.pkl'), 'wb') as f:
        pickle.dump(TransectInterGDFTopo, f)

In [ ]:
# Beach Width Plotting

# Select transect ID to plot
TransectIDs = [[25,30,35],50,75]
for TransectID in TransectIDs:
    # Plot timeseries of cross-shore width between water edge and veg edge 
    Plotting.WidthTimeseries(sitename, TransectInterGDFWater, TransectID, Hemisphere='N')

In [ ]:
# --- DEM / Slope unit inspector ---
# What it does:
# 1) Reads ./DEM.tif
# 2) Prints metadata + value stats
# 3) Heuristically classifies: elevation | slope_deg | slope_percent | slope_dzdx
# 4) Prints how to convert to S (dimensionless dz/dx) for normalization

import rasterio
import numpy as np
import json
from collections import Counter

TIF = "./DEM.tif"

def summarize_metadata(src):
    meta = src.meta.copy()
    tags = src.tags() or {}
    # Also collect band-specific tags if present
    band_tags = {}
    try:
        for b in range(1, src.count + 1):
            t = src.tags(b)
            if t:
                band_tags[f"band_{b}"] = t
    except Exception:
        pass

    # Common unit-ish clues
    all_tags = {**{k.lower(): v for k,v in tags.items()},
                **{(f"{k}".lower()): v for k,v in band_tags.items()}}
    clues = {}
    for k,v in all_tags.items():
        if any(key in k for key in ["unit", "units", "measure", "slope", "description", "long_name"]):
            clues[k] = v

    return meta, tags, band_tags, clues

def sample_stats(src, max_samples=500000):
    arr = src.read(1, masked=True)  # masked array (nodata masked)
    data = arr.compressed()  # drop masked nodata
    if data.size == 0:
        return {"count": 0}

    # Random subsample for speed if huge
    if data.size > max_samples:
        idx = np.random.choice(data.size, size=max_samples, replace=False)
        data = data[idx]

    finite = data[np.isfinite(data)]
    if finite.size == 0:
        return {"count": 0}

    stats = {
        "count": int(finite.size),
        "dtype": str(src.dtypes[0]),
        "min": float(np.min(finite)),
        "max": float(np.max(finite)),
        "p01": float(np.percentile(finite, 1)),
        "p50": float(np.percentile(finite, 50)),
        "p95": float(np.percentile(finite, 95)),
        "p99": float(np.percentile(finite, 99)),
        "neg_frac": float(np.mean(finite < 0)),
        "gt90_frac": float(np.mean(finite > 90)),
        "gt100_frac": float(np.mean(finite > 100)),
        "gt1000_frac": float(np.mean(finite > 1000)),
    }

    # A tiny histogram just for a feel
    hist, edges = np.histogram(finite, bins=10)
    stats["hist_edges"] = edges.tolist()
    stats["hist_counts"] = hist.tolist()

    # Pixel size
    tr = src.transform
    px = abs(tr.a)
    py = abs(tr.e)
    stats["pixel_size_x"] = float(px)
    stats["pixel_size_y"] = float(py)

    return stats

def classify(stats, clues):
    """
    Heuristics:
      - If negatives are common or very large values appear (>> 100), likely elevation (in meters).
      - If range 0..~90 and ~no values >90 -> slope in degrees.
      - If 0..~1000 and many values between ~1..200, maybe slope in percent.
      - If 0..~2 (rarely >3), looks like slope as dz/dx.
    """
    if not stats or stats.get("count", 0) == 0:
        return "unknown", "No valid data"

    vmin, vmax = stats["min"], stats["max"]
    p99 = stats["p99"]
    neg_frac = stats["neg_frac"]
    gt90 = stats["gt90_frac"]
    gt100 = stats["gt100_frac"]
    gt1000 = stats["gt1000_frac"]

    # Metadata clues first
    text = json.dumps(clues).lower()
    if any(s in text for s in ["degree", "degrees", "deg"]):
        guess = "slope_deg"
        reason = "Metadata mentions degree(s)."
        return guess, reason
    if any(s in text for s in ["percent", "%", "pct"]):
        guess = "slope_percent"
        reason = "Metadata mentions percent."
        return guess, reason
    if any(s in text for s in ["dz/dx", "slope_dzdx", "gradient"]):
        guess = "slope_dzdx"
        reason = "Metadata mentions dz/dx or gradient."
        return guess, reason
    if any(s in text for s in ["elev", "elevation", "height", "bathym", "depth"]):
        guess = "elevation"
        reason = "Metadata mentions elevation/depth."
        return guess, reason

    # Heuristics by distribution
    # Likely elevation: negatives or very large values common
    if neg_frac > 0.01 or gt1000 > 0.0 or vmax - vmin > 500:
        return "elevation", "Presence of negatives and/or very wide range suggests elevation (meters)."

    # Likely degrees
    if vmin >= 0 and gt90 < 1e-4 and vmax <= 90 and p99 <= 75:
        return "slope_deg", "Values confined to 0–90°, p99 ≤ ~75° -> degrees."

    # Likely percent
    if vmin >= 0 and vmax > 90 and p99 < 500 and gt1000 == 0:
        return "slope_percent", "Values >90 but <~500 common -> percent slope."

    # Likely dz/dx
    if vmin >= 0 and vmax <= 10 and p99 <= 3:
        return "slope_dzdx", "Values concentrated in 0–3 suggest dz/dx."

    # Fallback
    # Choose the closest bucket by p99
    if p99 <= 90:
        return "slope_deg", "Fallback by p99 ≤ 90."
    elif p99 <= 500:
        return "slope_percent", "Fallback by p99 ≤ 500."
    else:
        return "elevation", "Fallback due to very large dynamic range."

def print_recommendation(kind):
    print("\n=== Recommendation for tide normalization S (dz/dx) ===")
    if kind == "slope_deg":
        print("Detected: SLOPE IN DEGREES.")
        print("Use:      S = tan(deg * pi/180)")
    elif kind == "slope_percent":
        print("Detected: SLOPE IN PERCENT.")
        print("Use:      S = percent / 100.0")
    elif kind == "slope_dzdx":
        print("Detected: SLOPE AS DZ/DX (dimensionless).")
        print("Use:      S = value (no conversion).")
    elif kind == "elevation":
        print("Detected: ELEVATION (meters).")
        print("Action:   This is not slope. Compute slope from the DEM first.")
        print("          (I can give you a one-liner function to create slope_deg and slope_dzdx rasters.)")
    else:
        print("Unknown. Send me the stats printed above and I’ll decide.")

def main():
    with rasterio.open(TIF) as src:
        meta, tags, band_tags, clues = summarize_metadata(src)
        stats = sample_stats(src)

        print("=== Basic meta ===")
        print(json.dumps({k:str(v) for k,v in meta.items()}, indent=2))
        print("\n=== Useful metadata clues (searching for units/desc) ===")
        print(json.dumps(clues, indent=2))

        print("\n=== Value stats (after masking nodata) ===")
        print(json.dumps(stats, indent=2))

        kind, reason = classify(stats, clues)
        print(f"\n=== Classification: {kind} ===")
        print(f"Reason: {reason}")

        print_recommendation(kind)

        # Bonus: show what S would look like for the median value, by each interpretation
        if stats.get("count", 0) > 0:
            m = stats["p50"]
            S_deg = np.tan(np.deg2rad(m))
            S_pct = m / 100.0
            S_dzdx = m
            print("\n--- Sanity check using median value ---")
            print(f"Median value m = {m:.4f}")
            print(f"As degrees -> S = tan(m°)      = {S_deg:.4f}")
            print(f"As percent -> S = m/100        = {S_pct:.4f}")
            print(f"As dz/dx   -> S = m            = {S_dzdx:.4f}")
            print("(Pick the line matching the detected kind above.)")

if __name__ == "__main__":
    main()